# K Street Influence Tracker — Analysis Notebook

**Question:** Do lobbying clients who mention specific bills in their filings disproportionately target sponsors from competitive House districts?

This notebook uses the pipeline joins:
- LDA filings → extracted bill references
- Bill references → Congress.gov sponsors
- Sponsors → MIT Election Lab district results

It surfaces one finding and notes the caveats.

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DB_PATH = Path("../data/kstreet.duckdb")
con = duckdb.connect(str(DB_PATH), read_only=True)

print("Connected to", DB_PATH)

## 1. Load the joined data

In [ ]:
df = con.execute("""
    SELECT
        client_name,
        filing_year,
        bill_title,
        sponsor_name,
        sponsor_party,
        state_po,
        district,
        year_spend,
        two_party_margin,
        winner_party,
        competitiveness
    FROM mart_filing_sponsor_district
    WHERE chamber = 'House'
      AND year_spend > 0
      AND two_party_margin IS NOT NULL
""").df()

print(f"{len(df):,} filing-sponsor-district rows")
df.head()

## 2. Aggregate to client-bill level

One row per client-bill mention, with the sponsor's district competitiveness.

In [ ]:
client_bill = (
    df.groupby(["client_name", "bill_title", "sponsor_name", "state_po", "district", "competitiveness"])
    .agg(
        total_spend=("year_spend", "sum"),
        filings=("filing_year", "nunique"),
        avg_margin=("two_party_margin", "mean"),
    )
    .reset_index()
)

print(f"{len(client_bill):,} unique client-bill-sponsor combinations")
client_bill["competitiveness"].value_counts()

## 3. Finding: Spend concentration by district competitiveness

In [ ]:
spend_by_comp = (
    client_bill.groupby("competitiveness")
    .agg(
        total_spend=("total_spend", "sum"),
        mentions=("bill_title", "count"),
        unique_clients=("client_name", "nunique"),
        avg_margin=("avg_margin", "mean"),
    )
    .reset_index()
)

spend_by_comp["spend_share"] = spend_by_comp["total_spend"] / spend_by_comp["total_spend"].sum()
spend_by_comp["mention_share"] = spend_by_comp["mentions"] / spend_by_comp["mentions"].sum()

spend_by_comp.sort_values("total_spend", ascending=False)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
order = ["toss_up", "lean", "safe"]
plot_df = spend_by_comp.set_index("competitiveness").reindex(order).fillna(0)

x = range(len(order))
width = 0.35
ax.bar([i - width/2 for i in x], plot_df["spend_share"], width, label="Spend share", color="#1f77b4")
ax.bar([i + width/2 for i in x], plot_df["mention_share"], width, label="Mention share", color="#ff7f0e")

ax.set_xticks(x)
ax.set_xticklabels(["Toss-up\n(<5%)", "Lean\n(5–10%)", "Safe\n(>10%)"])
ax.set_ylabel("Share of total")
ax.set_title("Lobbying activity by sponsor district competitiveness")
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

## 4. Interpretation

**Finding:** [Update this sentence after running with real data.]

Example: *Clients mentioning specific bills spent a larger share of their reported dollars on sponsors from competitive districts than the raw mention count would suggest — indicating higher intensity (not just volume) on toss-up and lean races.*

### Caveats
- Bill references are extracted by regex from free text; informal mentions are missed.
- Only House sponsors have district-level results; Senate is excluded here.
- Competitiveness uses the nearest prior election, not a contemporaneous forecast.
- Reported spend is at the filing level, not allocated to individual bills.
- Sample is limited to bills that matched Congress.gov and had sponsor data pulled.

### Next steps
- Add issue-code filters (AI, energy, healthcare).
- Compare against a baseline of all House districts, not just those mentioned.
- Test whether the pattern holds after controlling for sponsor party and committee assignment.

In [ ]:
con.close()